In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

In [13]:
import wget
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain.vectorstores import FAISS
from langchain.llms import Ollama
import gradio as gr
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

In [3]:
# filename = 'companyPolicies.txt'
# url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# # Use wget to download the file
# wget.download(url, out=filename)
# print('file downloaded')

In [4]:
with open("../data/companyPolicies.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Wrap text in a Document object
docs = [Document(page_content=text)]

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = splitter.split_documents(docs)

print(f"Number of chunks: {len(split_docs)}")
print(split_docs[0].page_content[:200]) 

Number of chunks: 50
1.	Code of Conduct


In [6]:
embedding_model = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",  # compact, open-source, great for semantic search
    model_kwargs={"device": "cpu"},  # or "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)

/var/folders/m_/1qnj5yc165zbgt967t2m1nhh0000gn/T/ipykernel_4603/1910852058.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceBgeEmbeddings(


In [7]:
vectorstore = FAISS.from_documents(split_docs, embedding_model)
retriever = vectorstore.as_retriever()

In [8]:
llm = Ollama(model="gemma")  # or llama2, gemma, etc.


/var/folders/m_/1qnj5yc165zbgt967t2m1nhh0000gn/T/ipykernel_4603/1807639309.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma")  # or llama2, gemma, etc.


In [9]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
response = qa_chain.run("Summarize the document")
print(response)

/var/folders/m_/1qnj5yc165zbgt967t2m1nhh0000gn/T/ipykernel_4603/4008402208.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain.run("Summarize the document")


The provided document outlines the organization's policies and procedures related to creating a safe and inclusive work environment. It covers the following key aspects:

**Reporting Discrimination/Harassment:**
- Individuals can report incidents to their supervisor, manager, or HR representative.
- The organization investigates complaints promptly and confidentially.

**Policy Review:**
- The policy will be periodically reviewed for relevance and compliance with legal requirements.

**Selection Process:**
- Hiring is based on qualifications, experience, and skills, with objective interviews and assessments.

**Data Privacy:**
- Personal candidate information is protected and handled in accordance with data protection laws.

**Equal Opportunity:**
- The organization is an equal opportunity employer and does not discriminate based on protected statuses.

**Transparency:**
- Recruitment processes are transparent, with job postings advertised internally and externally.


In [10]:
def answer_question(question):
    return qa_chain.run(question)

interface = gr.Interface(fn=answer_question, inputs="text", outputs="text")
interface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Step 5: Build the Gradio interface
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)

with gr.Blocks() as demo:
    gr.Markdown("### 📞 Customer Support Chatbot (Gemma + FAISS)")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Ask a question...")
    
    def user_query(message, chat_history):
        result = qa_chain.run({"question": message, "chat_history": chat_history})
        chat_history.append((message, result))
        return "", chat_history

    msg.submit(user_query, [msg, chatbot], [msg, chatbot])

demo.launch()





/var/folders/m_/1qnj5yc165zbgt967t2m1nhh0000gn/T/ipykernel_4603/792879589.py:13: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

9.	Discipline and Termination Policy

This policy serves as a framework for handling discipline and termination. The organization recognizes the importance of fairness and consistency in these processes, and decisions will be made after careful consideration. Every employee is expected to understand and adhere to this policy, contributing to a respectful and productive workplace. Regular reviews will ensure its alignment with evolving legal requirements and best practices.

The Discipline and Termination Policy underscores the organization's commitment to maintaining a productive, ethical, and respectful work environment. This policy applies to all personnel, including employees, contractors, and temporary staff.

Conseq